# Gold Skill Demand Mart

**Purpose**: Dashboard-ready skill demand trends with aggregations.

**Target Table**: `workspace.gold.gold_skill_demand`

**Grain**: Date + sector/role/location + skill combinations

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation

**Key Metrics**:
- Skill mention counts
- Unique jobs requiring skills
- Average confidence scores

**Data Sources**:
- `workspace.warehouse.bridge_job_skill`
- `workspace.warehouse.fact_job_postings`

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_skill_demand_refresh_log`
- Captures: rows processed, unique dates/sectors/roles/locations/skills, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_skill_demand"
metadata_table = "workspace.metadata.gold_skill_demand_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")

In [0]:
%sql
-- Create table with all columns if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_skill_demand_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_dates INT,
  unique_sectors INT,
  unique_roles INT,
  unique_locations INT,
  unique_skills INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_skill_demand refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_skill_demand (
  demand_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT COMMENT 'FK to dim_sector (NULL for rollups)',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL for rollups)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL for rollups)',
  skill_sk BIGINT NOT NULL COMMENT 'FK to dim_skill',
  mentions_count BIGINT COMMENT 'Skill mention count',
  unique_jobs_count BIGINT COMMENT 'Unique jobs requiring skill',
  avg_confidence DECIMAL(10,4) COMMENT 'Average confidence score'
)
USING DELTA
COMMENT 'Skill demand trends with aggregations'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from datetime import datetime, timedelta

# Calculate cutoff date
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Building skill demand with aggregations (lookback: {lookback_days} days, cutoff: {cutoff_date})...")

# Insert aggregated skill demand data
insert_sql = f"""
INSERT OVERWRITE TABLE {target_table}

WITH skill_job_facts AS (
  SELECT
    f.posting_date_sk AS demand_date_sk,
    j.sector_sk,  -- Get sector from dim_job, not fact_job_postings
    f.role_sk,
    f.location_sk,
    bjs.skill_sk,
    bjs.job_sk,
    bjs.confidence
  FROM workspace.warehouse.bridge_job_skill bjs
  JOIN workspace.warehouse.fact_job_postings f ON bjs.job_sk = f.job_sk
  INNER JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= {cutoff_date}
    AND bjs.is_current = TRUE
    AND f.active_flag = TRUE
),

-- Sector rollup
sector_agg AS (
  SELECT
    demand_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    skill_sk,
    COUNT(*) AS mentions_count,
    COUNT(DISTINCT job_sk) AS unique_jobs_count,
    ROUND(AVG(confidence), 4) AS avg_confidence
  FROM skill_job_facts
  GROUP BY demand_date_sk, sector_sk, skill_sk
),

-- Role rollup
role_agg AS (
  SELECT
    demand_date_sk,
    CAST(NULL AS BIGINT),
    role_sk,
    CAST(NULL AS BIGINT),
    skill_sk,
    COUNT(*),
    COUNT(DISTINCT job_sk),
    ROUND(AVG(confidence), 4)
  FROM skill_job_facts
  GROUP BY demand_date_sk, role_sk, skill_sk
),

-- Location rollup
location_agg AS (
  SELECT
    demand_date_sk,
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    location_sk,
    skill_sk,
    COUNT(*),
    COUNT(DISTINCT job_sk),
    ROUND(AVG(confidence), 4)
  FROM skill_job_facts
  GROUP BY demand_date_sk, location_sk, skill_sk
)

SELECT * FROM sector_agg
UNION ALL
SELECT * FROM role_agg
UNION ALL
SELECT * FROM location_agg
"""

spark.sql(insert_sql)
print("✓ Data loaded")

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT demand_date_sk) as unique_dates,
            COUNT(DISTINCT sector_sk) as unique_sectors,
            COUNT(DISTINCT role_sk) as unique_roles,
            COUNT(DISTINCT location_sk) as unique_locations,
            COUNT(DISTINCT skill_sk) as unique_skills
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_dates = metrics['unique_dates']
    unique_sectors = metrics['unique_sectors']
    unique_roles = metrics['unique_roles']
    unique_locations = metrics['unique_locations']
    unique_skills = metrics['unique_skills']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Unique sectors: {unique_sectors}")
    print(f"✓ Unique roles: {unique_roles}")
    print(f"✓ Unique locations: {unique_locations}")
    print(f"✓ Unique skills: {unique_skills}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_dates = 0
    unique_sectors = 0
    unique_roles = 0
    unique_locations = 0
    unique_skills = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_dates,
            unique_sectors,
            unique_roles,
            unique_locations,
            unique_skills,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_dates},
            {unique_sectors},
            {unique_roles},
            {unique_locations},
            {unique_skills},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Overall summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT demand_date_sk) AS unique_dates,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT role_sk) AS unique_roles,
  COUNT(DISTINCT location_sk) AS unique_locations,
  COUNT(DISTINCT skill_sk) AS unique_skills,
  MIN(demand_date_sk) AS earliest_date,
  MAX(demand_date_sk) AS latest_date,
  SUM(mentions_count) AS total_mentions,
  SUM(unique_jobs_count) AS total_unique_jobs
FROM workspace.gold.gold_skill_demand;

In [0]:
%sql
-- Top skills by sector
SELECT
  ds.sector_name,
  dsk.skill_name,
  SUM(gsd.mentions_count) AS total_mentions,
  SUM(gsd.unique_jobs_count) AS total_unique_jobs,
  ROUND(AVG(gsd.avg_confidence), 4) AS avg_confidence
FROM workspace.gold.gold_skill_demand gsd
LEFT JOIN workspace.warehouse.dim_sector ds ON gsd.sector_sk = ds.sector_sk
LEFT JOIN workspace.warehouse.dim_skill dsk ON gsd.skill_sk = dsk.skill_sk
WHERE gsd.sector_sk IS NOT NULL
  AND gsd.role_sk IS NULL
  AND gsd.location_sk IS NULL
GROUP BY ds.sector_name, dsk.skill_name
ORDER BY total_mentions DESC
LIMIT 20;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null demand_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_skill_demand
WHERE demand_date_sk IS NULL

UNION ALL

SELECT
  'Null skill_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_skill_demand
WHERE skill_sk IS NULL

UNION ALL

SELECT
  'Negative mentions_count' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_skill_demand
WHERE mentions_count < 0

UNION ALL

SELECT
  'Invalid avg_confidence' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_skill_demand
WHERE avg_confidence < 0 OR avg_confidence > 1

ORDER BY issue_count DESC;

In [0]:
%sql
-- Breakdown by aggregation level
SELECT 
  CASE 
    WHEN sector_sk IS NOT NULL AND role_sk IS NULL AND location_sk IS NULL THEN 'Sector'
    WHEN sector_sk IS NULL AND role_sk IS NOT NULL AND location_sk IS NULL THEN 'Role'
    WHEN sector_sk IS NULL AND role_sk IS NULL AND location_sk IS NOT NULL THEN 'Location'
    ELSE 'Other'
  END AS rollup_level,
  COUNT(*) AS row_count,
  COUNT(DISTINCT demand_date_sk) AS unique_dates,
  COUNT(DISTINCT skill_sk) AS unique_skills,
  SUM(mentions_count) AS total_mentions,
  SUM(unique_jobs_count) AS total_unique_jobs
FROM workspace.gold.gold_skill_demand
GROUP BY rollup_level
ORDER BY row_count DESC;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_dates,
  unique_sectors,
  unique_roles,
  unique_locations,
  unique_skills,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_skill_demand_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;